# cesm extraction
daily surface shortwave and diffuse shortwave at the grid cell nearest each tower from cesm2 waccm6, arise g6 1.5k sai member 001 and ssp2 4.5 member 006, 2065 to 2084 (figure 6). runs where the cesm2 output is stored (ncar glade, casper)

In [ ]:
from pathlib import Path
import glob
import numpy as np, pandas as pd, xarray as xr

data = Path("data")
root = "/glade/campaign/cesm/collections"
runs = {"SAI": [f"{root}/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.001"],
        "SSP": [f"{root}/CESM2-WACCM-SSP245/b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.006",
                f"{root}/CESM2-WACCM-SSP245/b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.006ext"]}
sites = {"US-Bi2": (38.1091, -121.5351), "US-Ne1": (41.1651, -96.4766), "US-Ne2": (41.1649, -96.4701),
         "US-Ne3": (41.1797, -96.4397), "US-xUN": (46.2339, -89.5373), "US-UMB": (45.5598, -84.7138),
         "US-MMS": (39.3232, -86.4131), "US-Ha1": (42.5378, -72.1715)}

In [ ]:
def at_sites(files, var):
    out = []
    for f in files:
        with xr.open_dataset(f) as ds:
            # date of each day from the start of its averaging interval
            bnd = next((b for b in ("time_bnds", "time_bounds") if b in ds), None)
            t = ds[bnd].isel({ds[bnd].dims[-1]: 0}).values if bnd else ds.time.values
            t = pd.to_datetime([f"{x.year:04d}-{x.month:02d}-{x.day:02d}" for x in t])
            keep = np.asarray((t >= "2065-01-01") & (t <= "2084-12-31"))
            if not keep.any():
                continue
            for s, (la, lo) in sites.items():
                i, j = int(np.abs(ds.lat.values - la).argmin()), int(np.abs(ds.lon.values - lo % 360).argmin())
                out.append(pd.DataFrame({"date": t[keep], "site": s, var: ds[var].isel(lat=i, lon=j).values[keep]}))
    return pd.concat(out).drop_duplicates(["date", "site"])


tabs = []
for run, dirs in runs.items():
    tab = None
    for var in ("FSDS", "SOLSD", "SOLLD"):
        files = sorted(f for d in dirs for f in glob.glob(f"{d}/atm/proc/tseries/day_1/*.cam.h1.{var}.*.nc"))
        v = at_sites(files, var)
        tab = v if tab is None else tab.merge(v, on=["date", "site"], how="outer")
    tabs.append(tab.assign(run=run))
pd.concat(tabs).sort_values(["run", "site", "date"]).to_csv(data / "cesm_daily.csv", index=False)